<a href="https://colab.research.google.com/github/HarshiniChebrolu/Flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshiniChebrolu/Flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

The FlyRank research paper reports findings about AI-driven SEO based on its research methodology and collected observations. For this audit, I selected two findings and focused on the methodology questions I would ask before extending those findings to broader conclusions.

### Finding 1

The paper reports that AI-generated answers can affect how users discover and interact with information from search results.

**My methodology question:**  
How were these changes in user discovery or interaction measured, and what was the source and definition of the underlying labels or outcome measures? I would want to understand whether the measurements came from directly observed behavior, a defined dataset, or another measurement process before interpreting the finding beyond the studied sample.

### Finding 2

The paper reports differences in how websites or content perform in AI-driven search environments.

**My methodology question:**  
Does the validation or comparison design support extending this observed difference to other websites, clients, or time periods? In particular, I would want to know whether the observations were separated by client or other independent groups, and whether the evaluation design prevents related observations from influencing both sides of the comparison.

These questions are intended as constructive methodology checks. They do not invalidate the reported findings; they identify what I would want to understand about measurement, labeling, and validation before making a broader claim.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 2. My model under an honest split (before/after)

### Before: random stratified split

In Week 5, the model was evaluated using a stratified 80/20 train-test split. This preserves the observed class proportions, but it does not explicitly prevent related observations from the same client from appearing in both training and test sets.

This can make the test observations less independent from the training observations when multiple rows belong to the same client.

### After: grouped split

For this validation audit, I will use a client-grouped split where observations from the same client are kept in the same partition.

The purpose is to test whether the measured model performance remains similar when the model is evaluated on clients that were not represented in its training data.

The comparison is descriptive: it shows how the measured performance changes under a stricter validation design. It does not establish that the model will perform the same way on every future client.

In [5]:
# ML-09 — Honest client-grouped split

group_col = "client_id"

if group_col not in model_df.columns:
    raise KeyError(f"{group_col} is not available.")

# Remove rows with missing client IDs
grouped_df = model_df[
    model_df[group_col].notna()
].copy()

X_group = grouped_df[feature_cols].copy()
y_group = grouped_df[target_col].copy()
groups = grouped_df[group_col].copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_group, y_group, groups=groups)
)

X_group_train = X_group.iloc[train_idx].copy()
X_group_test = X_group.iloc[test_idx].copy()

y_group_train = y_group.iloc[train_idx].copy()
y_group_test = y_group.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_group_train))
print("Test rows:", len(X_group_test))

print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

overlap = set(groups_train.unique()).intersection(
    set(groups_test.unique())
)

print("Client overlap:", len(overlap))

if len(overlap) != 0:
    raise ValueError("Client leakage detected.")

print("✓ No client appears in both train and test.")

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
✓ No client appears in both train and test.


In [6]:
# Preprocessing — same approach as Week 5

numeric_features = X_group_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_group_train.select_dtypes(
    exclude=["int64", "float64"]
).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

grouped_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

grouped_model.fit(
    X_group_train,
    y_group_train
)

grouped_pred = grouped_model.predict(
    X_group_test
)

grouped_accuracy = accuracy_score(
    y_group_test,
    grouped_pred
)

grouped_macro_f1 = f1_score(
    y_group_test,
    grouped_pred,
    average="macro"
)

print("Client-grouped validation")
print("-------------------------")
print("Accuracy:", round(grouped_accuracy, 4))
print("Macro F1:", round(grouped_macro_f1, 4))

Client-grouped validation
-------------------------
Accuracy: 0.6568
Macro F1: 0.5097


In [7]:
# Before vs After comparison

# Week-5 random split
X_random_train, X_random_test, y_random_train, y_random_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

random_numeric = X_random_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

random_categorical = X_random_train.select_dtypes(
    exclude=["int64", "float64"]
).columns.tolist()

random_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), random_numeric),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), random_categorical)
])

random_model = Pipeline([
    ("preprocessor", random_preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

random_model.fit(X_random_train, y_random_train)

random_pred = random_model.predict(X_random_test)

random_accuracy = accuracy_score(
    y_random_test,
    random_pred
)

random_macro_f1 = f1_score(
    y_random_test,
    random_pred,
    average="macro"
)

comparison = pd.DataFrame({
    "Validation design": [
        "Week-5 random stratified split",
        "ML-09 client-grouped split"
    ],
    "Accuracy": [
        random_accuracy,
        grouped_accuracy
    ],
    "Macro F1": [
        random_macro_f1,
        grouped_macro_f1
    ]
})

print(comparison.to_string(index=False))

             Validation design  Accuracy  Macro F1
Week-5 random stratified split  0.757000  0.664402
    ML-09 client-grouped split  0.656823  0.509735




## 3. Leakage audit

I audited the final feature set for columns that could directly reveal the target or represent information that would only be known after the outcome.

The target is `trend_direction`. The Week-5 modeling setup excludes `trend_direction` itself, along with `trend_pct`, `action`, and `action_score`, because these fields are outcome-related and should not be used as predictive inputs.

I also excluded identifier fields such as `client_id`, `content_id`, `url`, `page_url`, and `client_name` from the model features where present. `client_id` is used only to create the grouped validation split, not as a predictive feature.

The audit below checks that these excluded fields are not present in the final feature set.

In [2]:
# Section 3 — Leakage Audit
# Load the dataset independently so this section can run by itself

import pandas as pd

# Upload dataset
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))
df_audit = pd.read_csv(filename)

print("Dataset shape:", df_audit.shape)

# Target and fields that should never be model inputs
target_col = "trend_direction"

excluded_cols = [
    "trend_direction",
    "trend_pct",
    "action",
    "action_score",
    "client_id",
    "content_id",
    "url",
    "page_url",
    "client_name"
]

# Create final feature list
audit_feature_cols = [
    col for col in df_audit.columns
    if col not in excluded_cols
]

print("\nNumber of final model features:", len(audit_feature_cols))

# Leakage candidates
leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "action",
    "action_score"
]

identifier_candidates = [
    "client_id",
    "content_id",
    "url",
    "page_url",
    "client_name"
]

# Check for leakage
outcome_in_features = [
    col for col in leakage_candidates
    if col in audit_feature_cols
]

identifiers_in_features = [
    col for col in identifier_candidates
    if col in audit_feature_cols
]

print("\n=== FINAL LEAKAGE AUDIT ===")

print("\nOutcome-related columns found in model features:")
print(outcome_in_features)

print("\nIdentifier columns found in model features:")
print(identifiers_in_features)

# Final checks
assert target_col not in audit_feature_cols
assert not any(
    col in audit_feature_cols
    for col in ["trend_pct", "action", "action_score"]
)
assert "client_id" not in audit_feature_cols
assert "content_id" not in audit_feature_cols

print("\n✓ Target and outcome-related fields are excluded.")
print("✓ Identifier columns are excluded from prediction.")
print("✓ client_id is reserved for grouped validation.")

Saving content_refresh_anonymized (2).csv to content_refresh_anonymized (2).csv
Dataset shape: (30000, 44)

Number of final model features: 40

=== FINAL LEAKAGE AUDIT ===

Outcome-related columns found in model features:
[]

Identifier columns found in model features:
[]

✓ Target and outcome-related fields are excluded.
✓ Identifier columns are excluded from prediction.
✓ client_id is reserved for grouped validation.


## 4. Claim rewrite

### Original claim

The Week-5 model can predict whether a page will decline, recover, or gain momentum.

### Safer claim

In the evaluated dataset, the Logistic Regression model **measured** predictive performance for the `trend_direction` classes under both a random stratified split and a client-grouped split.

The **observed** results show how Accuracy and Macro F1 changed when validation was performed on clients that were not represented in the training partition.

These results are **directional** and provide **decision-support** for evaluating the modeling approach. They should not be interpreted as proof that the model will perform identically on every future client or dataset.

In [4]:
# Section 4 — Claim rewrite check

print("=== CLAIM REWRITE ===")

print("Original claim:")
print(
    "The Week-5 model can predict whether a page will decline, "
    "recover, or gain momentum."
)

print("\nSafer interpretation:")
print(
    "The evaluated model measured predictive performance for "
    "trend_direction in the available dataset."
)

print(
    "\nThe results are directional and provide decision-support "
    "for evaluating the modeling approach."
)

print(
    "\nThe results should not be interpreted as proof that the "
    "model will perform identically on every future client or dataset."
)

=== CLAIM REWRITE ===
Original claim:
The Week-5 model can predict whether a page will decline, recover, or gain momentum.

Safer interpretation:
The evaluated model measured predictive performance for trend_direction in the available dataset.

The results are directional and provide decision-support for evaluating the modeling approach.

The results should not be interpreted as proof that the model will perform identically on every future client or dataset.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.